# 🎯 Quranic Qira'at ML - Quick Start Tutorial

**Date**: March 1, 2026  
**Status**: Production Ready  
**Hardware**: RTX 5070 Ti (12GB VRAM)

This notebook walks you through the complete pipeline in 15 minutes.

## Step 1: Setup & Imports

In [ ]:
# Install dependencies (run once)
# !pip install -r ../requirements.txt

In [ ]:
import sys
from pathlib import Path

# Add src to path
sys.path.insert(0, str(Path.cwd() / 'src'))

import torch
import numpy as np
from src.preprocess import prepare_dataset, load_audio_safe, normalize_audio_level
from src.tajweed_rules import TajweedDetector
from src.train import QiraatMTLModel, TrainingConfig

print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f}GB")

## Step 2: Verify Data Setup

In [ ]:
from pathlib import Path

# Check if data directories exist
data_dir = Path('./data')
raw_dir = data_dir / 'raw'
metadata_dir = data_dir / 'metadata'

print("📁 Directory Structure:")
print(f"  Raw data: {raw_dir} - {'✓' if raw_dir.exists() else '✗'}")
print(f"  Metadata: {metadata_dir} - {'✓' if metadata_dir.exists() else '✗'}")

# Check for audio files
if raw_dir.exists():
    hafs_files = list((raw_dir / 'hafs').glob('*.wav')) if (raw_dir / 'hafs').exists() else []
    warsh_files = list((raw_dir / 'warsh').glob('*.wav')) if (raw_dir / 'warsh').exists() else []
    print(f"  Hafs samples: {len(hafs_files)}")
    print(f"  Warsh samples: {len(warsh_files)}")

## Step 3: Load Sample Audio

In [ ]:
from src.preprocess import load_audio_safe, normalize_audio_level
import librosa.display
import matplotlib.pyplot as plt

# Load sample audio (replace with your path)
sample_audio_path = Path('./data/raw/hafs/001_001_hafs.wav')

if sample_audio_path.exists():
    print(f"Loading: {sample_audio_path.name}")
    audio, sr = load_audio_safe(sample_audio_path, target_sr=16000)
    
    # Normalize
    audio_normalized = normalize_audio_level(audio, target_db=-20.0)
    
    print(f"Audio shape: {audio_normalized.shape}")
    print(f"Sample rate: {sr}Hz")
    print(f"Duration: {len(audio_normalized) / sr:.2f}s")
    print(f"Loudness: -20dB (normalized)")
    
    # Plot waveform
    plt.figure(figsize=(12, 4))
    librosa.display.waveshow(audio_normalized, sr=sr)
    plt.title('Audio Waveform')
    plt.show()
else:
    print(f"⚠️  File not found: {sample_audio_path}")
    print("Please place audio files in: ./data/raw/hafs/ or ./data/raw/warsh/")

## Step 4: Detect Tajweed Rules

In [ ]:
from src.tajweed_rules import TajweedDetector, tajweed_score_to_json
import json

# Initialize detector
detector = TajweedDetector(sample_rate=16000)

# Score Tajweed
if 'audio_normalized' in locals():
    print("🔍 Analyzing Tajweed Rules...")
    score = detector.score_tajweed(
        waveform=audio_normalized,
        predicted_qiraat='Hafs',
        expected_qiraat='Hafs',
        qiraat_confidence=0.92,
    )
    
    # Export as JSON
    result = tajweed_score_to_json(score)
    print(json.dumps(result, indent=2, ensure_ascii=False))
else:
    print("⚠️  No audio loaded. Please load audio first.")

## Step 5: Prepare Dataset

In [ ]:
from src.preprocess import prepare_dataset
from pathlib import Path

print("⏳ Preparing dataset (this may take 10-30 minutes)...")

dataset_dict = prepare_dataset(
    raw_data_dir=Path('./data/raw'),
    metadata_manifest=Path('./data/metadata/hafs_manifest.json'),
    output_dir=Path('./data/processed'),
    cache_dir=Path('./data/cache'),
    apply_augmentation=True,
)

print(f"\n✅ Dataset prepared!")
print(f"  Train: {len(dataset_dict['train'])} samples")
print(f"  Validation: {len(dataset_dict['validation'])} samples")
print(f"  Test: {len(dataset_dict['test'])} samples")

## Step 6: Initialize Model

In [ ]:
from src.train import QiraatMTLModel
import torch

print("🔧 Initializing Multi-Task Learning Model...")

model = QiraatMTLModel(
    model_name="facebook/wav2vec2-xlsr-128d",
    num_qiraat_classes=2,  # Hafs, Warsh
    num_tajweed_rules=6,
    lora_rank=8,
    lora_alpha=32,
    use_gradient_checkpointing=True,
)

# Move to GPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)

print(f"✅ Model initialized on {device}")

# Check trainable parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"  Total params: {total_params:,}")
print(f"  Trainable params: {trainable_params:,} ({100*trainable_params/total_params:.1f}%)")

## Step 7: Check VRAM Usage

In [ ]:
if torch.cuda.is_available():
    print("💾 VRAM Usage:")
    allocated = torch.cuda.memory_allocated() / 1e9
    reserved = torch.cuda.memory_reserved() / 1e9
    total = torch.cuda.get_device_properties(0).total_memory / 1e9
    
    print(f"  Allocated: {allocated:.2f}GB")
    print(f"  Reserved: {reserved:.2f}GB")
    print(f"  Total: {total:.2f}GB")
    print(f"  Available: {total - allocated:.2f}GB")
    
    if allocated < total * 0.8:
        print(f"\n✅ Safe VRAM usage (within limits)")
    else:
        print(f"\n⚠️  High VRAM usage (>80%)")

## Next Steps

1. **Prepare your data**: Place audio files in `./data/raw/hafs/` and `./data/raw/warsh/`
2. **Create metadata**: Create JSON files in `./data/metadata/`
3. **Run preprocessing**: `python src/preprocess.py`
4. **Start training**: `python src/train.py`
5. **Monitor**: Open Weights & Biases dashboard

See `notebooks/2_training.ipynb` for detailed training walkthrough.